# Capítulo 10: Monitoreando en Producción (Data Drift y Concept Drift)

> *"Tu modelo es como un mapamundi — se queda obsoleto sin que te des cuenta."*

## Mandamiento 4: El infierno que viene después del notebook

In [ ]:
# Celda 1: Importaciones

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
sns.set_palette("husl")

# Configuración de pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("✅ Importaciones completadas")
print(f"📅 Fecha: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## Celda 2: Simulación de datos originales

Simularemos un dataset de ventas con features que simulan el comportamiento original del modelo.

In [ ]:
# Celda 2: Simulación de datos originales

np.random.seed(42)
n_muestras = 5000

# Generar timestamps
fechas = pd.date_range(start='2024-01-01', periods=n_muestras, freq='H')

# Features originales (distribuciones base)
feature_1 = np.random.normal(100, 15, n_muestras)  # Ej: temperatura promedio
feature_2 = np.random.uniform(10, 100, n_muestras)  # Ej: precio del producto
feature_3 = np.random.exponential(5, n_muestras)     # Ej: tiempo de espera

# Variable objetivo (relación original: lineal con feature_2)
# Regla: feature_2 alta → mayor probabilidad de compra
probabilidad_compra = 1 / (1 + np.exp(-(0.05 * (feature_2 - 50) + np.random.normal(0, 0.5, n_muestras))))
target = np.random.binomial(1, probabilidad_compra)

# Predicciones del modelo (buen rendimiento inicial)
prediction = target.copy()  # Modelo perfecto en entrenamiento
prediction[np.random.choice(n_muestras, size=200, replace=False)] = 1 - prediction[np.random.choice(n_muestras, size=200, replace=False)]  # ~4% error

# Probabilidades del modelo
probability = probabilidad_compra + np.random.normal(0, 0.1, n_muestras)
probability = np.clip(probability, 0, 1)

# Crear DataFrame
df_original = pd.DataFrame({
    'timestamp': fechas,
    'feature_1': feature_1,
    'feature_2': feature_2,
    'feature_3': feature_3,
    'target': target,
    'prediction': prediction,
    'probability': probability
})

print("📊 Datos originales creados:")
print(f"   • Muestras: {len(df_original)}")
print(f"   • Período: {df_original['timestamp'].min()} a {df_original['timestamp'].max()}")
print(f"   • Tasa de compra: {df_original['target'].mean():.2%}")
print(f"   • Accuracy del modelo: {(df_original['target'] == df_original['prediction']).mean():.2%}")
print("\n📈 Primeras filas:")
df_original.head()

## Celda 3: Simulación de drift

Ahora simularemos **data drift** (cambio en feature_1) y **concept drift** (cambio en la relación feature_2 → target).

In [ ]:
# Celda 3: Simulación de drift

np.random.seed(123)
n_muestras_drift = 5000

# Timestamps para período de drift
fechas_drift = pd.date_range(start='2024-07-01', periods=n_muestras_drift, freq='H')

# DATA DRIFT: feature_1 cambia gradualmente
# Media se desplaza de 100 → 120, desviación de 15 → 20
drift_factor = np.linspace(0, 1, n_muestras_drift)
feature_1_drift = (100 + 20 * drift_factor) + np.random.normal(0, 15 + 5 * drift_factor, n_muestras_drift)

# feature_2 y feature_3 permanecen iguales
feature_2_drift = np.random.uniform(10, 100, n_muestras_drift)
feature_3_drift = np.random.exponential(5, n_muestras_drift)

# CONCEPT DRIFT: la relación feature_2 → target CAMBIA
# Antes: feature_2 alta → compra probable
# Ahora: feature_2 alta → compra MENOS probable (reglas del juego cambian)
probabilidad_compra_drift = 1 / (1 + np.exp(-(-0.05 * (feature_2_drift - 50) + np.random.normal(0, 0.5, n_muestras_drift))))
target_drift = np.random.binomial(1, probabilidad_compra_drift)

# El modelo viejo sigue prediciendo con la regla antigua
prob_modelo_viejo = 1 / (1 + np.exp(-(0.05 * (feature_2_drift - 50) + np.random.normal(0, 0.5, n_muestras_drift))))
prediction_drift = (prob_modelo_viejo > 0.5).astype(int)

# Probabilidades del modelo viejo
probability_drift = prob_modelo_viejo

# Crear DataFrame con drift
df_drift = pd.DataFrame({
    'timestamp': fechas_drift,
    'feature_1': feature_1_drift,
    'feature_2': feature_2_drift,
    'feature_3': feature_3_drift,
    'target': target_drift,
    'prediction': prediction_drift,
    'probability': probability_drift
})

print("🔴 Datos con DRIFT creados:")
print(f"   • Muestras: {len(df_drift)}")
print(f"   • Período: {df_drift['timestamp'].min()} a {df_drift['timestamp'].max()}")
print(f"   • Nueva tasa de compra: {df_drift['target'].mean():.2%}")
print(f"   • Accuracy del modelo viejo: {(df_drift['target'] == df_drift['prediction']).mean():.2%}")
print("\n⚠️  CAMBIOS DETECTADOS:")
print(f"   • feature_1: Media {df_original['feature_1'].mean():.1f} → {df_drift['feature_1'].mean():.1f}")
print(f"   • Relación feature_2→target: INVERTIDA")
print(f"   • Caída de accuracy: {((df_original['target'] == df_original['prediction']).mean() - (df_drift['target'] == df_drift['prediction']).mean())*100:.1f}%")

## Celda 4: Detección de Drift con KS Test

El test de Kolmogorov-Smirnov compara si dos muestras provienen de la misma distribución.

In [ ]:
# Celda 4: Detección con KS Test

def detectar_drift_ks(datos_entrenamiento, datos_produccion, alpha=0.05):
    """
    Detecta data drift usando el test Kolmogorov-Smirnov.
    """
    stat, p_value = stats.ks_2samp(datos_entrenamiento, datos_produccion)
    
    if p_value < 0.01:
        severidad = 'CRÍTICA'
    elif p_value < 0.05:
        severidad = 'ALTA'
    elif p_value < 0.1:
        severidad = 'MEDIA'
    else:
        severidad = 'BAJA'
    
    return {
        'drift_detectado': p_value < alpha,
        'estadistico_ks': stat,
        'p_valor': p_value,
        'severidad': severidad
    }

# Aplicar KS Test a cada feature
print("="*60)
print("TEST DE KOLMOGOROV-SMIRNOV")
print("="*60)

resultados_ks = {}
features = ['feature_1', 'feature_2', 'feature_3']

for feature in features:
    resultado = detectar_drift_ks(
        df_original[feature], 
        df_drift[feature]
    )
    resultados_ks[feature] = resultado
    
    emoji = '🔴' if resultado['drift_detectado'] else '🟢'
    print(f"\n{emoji} {feature.upper()}:")
    print(f"   • Estadístico KS: {resultado['estadistico_ks']:.4f}")
    print(f"   • P-valor: {resultado['p_valor']:.6f}")
    print(f"   • Severidad: {resultado['severidad']}")
    print(f"   • Drift detectado: {'SÍ' if resultado['drift_detectado'] else 'NO'}")

# Resumen
features_con_drift = [f for f, r in resultados_ks.items() if r['drift_detectado']]
print(f"\n{'='*60}")
print(f"RESUMEN: {len(features_con_drift)}/{len(features)} features con drift significativo")
if features_con_drift:
    print(f"Features con drift: {', '.join(features_con_drift)}")

## Celda 5: Detección de Drift con PSI

El Population Stability Index (PSI) mide el cambio en la distribución de una variable.

- PSI < 0.1: Sin drift significativo
- PSI 0.1-0.2: Drift moderado
- PSI > 0.2: Drift significativo

In [ ]:
# Celda 5: Detección con PSI

def calcular_psi(datos_entrenamiento, datos_produccion, bins=10):
    """
    Calcula el Population Stability Index (PSI).
    """
    # Crear bins basados en percentiles de datos de entrenamiento
    breakpoints = np.percentile(datos_entrenamiento, 
                                 np.linspace(0, 100, bins + 1))
    breakpoints[0] = -np.inf
    breakpoints[-1] = np.inf
    
    # Evitar duplicados en breakpoints
    breakpoints = np.unique(breakpoints)
    
    # Calcular proporciones en cada bin
    hist_entrenamiento, _ = np.histogram(datos_entrenamiento, bins=breakpoints)
    hist_produccion, _ = np.histogram(datos_produccion, bins=breakpoints)
    
    # Normalizar con suavizado para evitar división por cero
    prop_entrenamiento = (hist_entrenamiento + 1) / (len(datos_entrenamiento) + bins)
    prop_produccion = (hist_produccion + 1) / (len(datos_produccion) + bins)
    
    # Calcular PSI
    psi = np.sum((prop_produccion - prop_entrenamiento) * 
                 np.log(prop_produccion / prop_entrenamiento))
    
    # Clasificar severidad
    if psi > 0.25:
        severidad = 'CRÍTICA'
    elif psi > 0.1:
        severidad = 'MODERADA'
    else:
        severidad = 'BAJA'
    
    return {
        'psi': psi,
        'severidad': severidad,
        'drift_detectado': psi > 0.1,
        'drift_significativo': psi > 0.2
    }

# Aplicar PSI a cada feature
print("="*60)
print("POPULATION STABILITY INDEX (PSI)")
print("="*60)

resultados_psi = {}

for feature in features:
    resultado = calcular_psi(
        df_original[feature], 
        df_drift[feature]
    )
    resultados_psi[feature] = resultado
    
    if resultado['drift_significativo']:
        emoji = '🔴'
    elif resultado['drift_detectado']:
        emoji = '🟡'
    else:
        emoji = '🟢'
    
    print(f"\n{emoji} {feature.upper()}:")
    print(f"   • PSI: {resultado['psi']:.4f}")
    print(f"   • Severidad: {resultado['severidad']}")

# Comparación KS vs PSI
print(f"\n{'='*60}")
print("COMPARACIÓN KS vs PSI")
print("="*60)
print(f"{'Feature':<15} {'KS Drift':<12} {'PSI Drift':<12} {'Concordancia':<12}")
print("-"*50)

for feature in features:
    ks_drift = 'SÍ' if resultados_ks[feature]['drift_detectado'] else 'NO'
    psi_drift = 'SÍ' if resultados_psi[feature]['drift_detectado'] else 'NO'
    concordancia = '✅' if ks_drift == psi_drift else '⚠️'
    print(f"{feature:<15} {ks_drift:<12} {psi_drift:<12} {concordancia:<12}")

## Celda 6: Visualización de Drift

Visualizaremos el drift para entender mejor los cambios en las distribuciones.

In [ ]:
# Celda 6: Visualización de drift

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Detección de Data Drift: Datos Originales vs Producción', 
             fontsize=16, fontweight='bold', y=1.02)

# Fila 1: Histogramas comparativos
for idx, feature in enumerate(features):
    ax = axes[0, idx]
    
    # Histogramas superpuestos
    ax.hist(df_original[feature], bins=30, alpha=0.5, label='Original', 
            color='steelblue', density=True)
    ax.hist(df_drift[feature], bins=30, alpha=0.5, label='Con Drift', 
            color='coral', density=True)
    
    # Estadísticas
    mean_orig = df_original[feature].mean()
    mean_drift = df_drift[feature].mean()
    
    ax.axvline(mean_orig, color='steelblue', linestyle='--', linewidth=2, 
               label=f'Media Orig: {mean_orig:.1f}')
    ax.axvline(mean_drift, color='coral', linestyle='--', linewidth=2, 
               label=f'Media Drift: {mean_drift:.1f}')
    
    ax.set_title(f'{feature.upper()}\nPSI = {resultados_psi[feature]["psi"]:.3f}', 
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Valor')
    ax.set_ylabel('Densidad')
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(True, alpha=0.3)

# Fila 2: Evolución temporal y métricas
for idx, feature in enumerate(features):
    ax = axes[1, idx]
    
    # Media móvil en ventanas de 100 muestras
    window_size = 100
    media_movil_orig = df_original[feature].rolling(window=window_size).mean()
    media_movil_drift = df_drift[feature].rolling(window=window_size).mean()
    
    # Graficar
    ax.plot(range(len(media_movil_orig)), media_movil_orig, 
            label='Original', color='steelblue', linewidth=2)
    ax.plot(range(len(media_movil_orig), len(media_movil_orig) + len(media_movil_drift)), 
            media_movil_drift, label='Con Drift', color='coral', linewidth=2)
    
    # Línea de separación
    ax.axvline(x=len(media_movil_orig), color='gray', linestyle=':', 
               linewidth=2, label='Inicio Drift')
    
    ax.set_title(f'Evolución Temporal: {feature.upper()}', 
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Índice de Muestra')
    ax.set_ylabel('Media Móvil')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Gráfico adicional: Caída de performance
fig, ax = plt.subplots(figsize=(10, 6))

accuracy_original = (df_original['target'] == df_original['prediction']).mean()
accuracy_drift = (df_drift['target'] == df_drift['prediction']).mean()

barras = ax.bar(['Original', 'Con Drift'], 
                [accuracy_original * 100, accuracy_drift * 100],
                color=['steelblue', 'coral'], edgecolor='black', linewidth=1.5)

# Añadir valores en las barras
for bar, valor in zip(barras, [accuracy_original * 100, accuracy_drift * 100]):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
            f'{valor:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=14)

ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Caída de Performance por Concept Drift', fontsize=14, fontweight='bold')
ax.set_ylim(0, 105)
ax.grid(True, alpha=0.3, axis='y')

# Añadir anotación
caida = (accuracy_original - accuracy_drift) * 100
ax.annotate(f'Caída: {caida:.1f}%', xy=(1, accuracy_drift * 100),
            xytext=(1.2, (accuracy_original + accuracy_drift) / 2 * 100),
            arrowprops=dict(arrowstyle='->', color='red', lw=2),
            fontsize=12, color='red', fontweight='bold')

plt.tight_layout()
plt.show()

## Celda 7: Sistema de Alertas

Implementaremos un sistema de alertas automatizado que detecta drift y notifica a los equipos.

In [ ]:
# Celda 7: Sistema de alertas

class SistemaAlertas:
    """
    Sistema de alertas para monitoreo de modelos en producción.
    """
    
    def __init__(self, umbrales=None):
        self.umbrales = umbrales or {
            'psi_critico': 0.2,
            'psi_moderado': 0.1,
            'accuracy_minimo': 0.7,
            'accuracy_advertencia': 0.8,
            'max_nulos_porcentaje': 0.1
        }
        self.historial_alertas = []
    
    def evaluar_drift(self, resultados_psi, resultados_ks):
        """
        Evalúa resultados de drift y genera alertas.
        """
        alertas = []
        
        # Verificar PSI para cada feature
        for feature, resultado in resultados_psi.items():
            if resultado['psi'] > self.umbrales['psi_critico']:
                alertas.append({
                    'tipo': 'DRIFT_CRITICO',
                    'severidad': 'CRÍTICA',
                    'feature': feature,
                    'mensaje': f"PSI = {resultado['psi']:.3f} > {self.umbrales['psi_critico']}",
                    'timestamp': datetime.now().isoformat(),
                    'accion': 'Reentrenamiento inmediato requerido'
                })
            elif resultado['psi'] > self.umbrales['psi_moderado']:
                alertas.append({
                    'tipo': 'DRIFT_MODERADO',
                    'severidad': 'MEDIA',
                    'feature': feature,
                    'mensaje': f"PSI = {resultado['psi']:.3f}",
                    'timestamp': datetime.now().isoformat(),
                    'accion': 'Monitoreo intensivo recomendado'
                })
        
        return alertas
    
    def evaluar_accuracy(self, accuracy_actual):
        """
        Evalúa la precisión del modelo.
        """
        alertas = []
        
        if accuracy_actual < self.umbrales['accuracy_minimo']:
            alertas.append({
                'tipo': 'RENDIMIENTO_CRITICO',
                'severidad': 'CRÍTICA',
                'mensaje': f"Accuracy = {accuracy_actual:.3f} < {self.umbrales['accuracy_minimo']}",
                'timestamp': datetime.now().isoformat(),
                'accion': 'Rollback inmediato necesario'
            })
        elif accuracy_actual < self.umbrales['accuracy_advertencia']:
            alertas.append({
                'tipo': 'RENDIMIENTO_ADVERTENCIA',
                'severidad': 'ALTA',
                'mensaje': f"Accuracy = {accuracy_actual:.3f}",
                'timestamp': datetime.now().isoformat(),
                'accion': 'Investigar causa de degradación'
            })
        
        return alertas
    
    def generar_reporte(self, resultados_psi, accuracy_actual):
        """
        Genera reporte completo de alertas.
        """
        alertas_drift = self.evaluar_drift(resultados_psi, {})
        alertas_accuracy = self.evaluar_accuracy(accuracy_actual)
        
        todas_alertas = alertas_drift + alertas_accuracy
        self.historial_alertas.extend(todas_alertas)
        
        print("="*60)
        print("🚨 REPORTE DE ALERTAS")
        print("="*60)
        print(f"📅 Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"📊 Accuracy actual: {accuracy_actual:.2%}")
        print(f"\n🔔 Total de alertas: {len(todas_alertas)}")
        
        if todas_alertas:
            print("\n" + "-"*60)
            for i, alerta in enumerate(todas_alertas, 1):
                emoji = '🔴' if alerta['severidad'] == 'CRÍTICA' else '🟡' if alerta['severidad'] == 'ALTA' else '🟠'
                print(f"\n{emoji} Alerta {i}:")
                print(f"   Tipo: {alerta['tipo']}")
                print(f"   Severidad: {alerta['severidad']}")
                print(f"   Mensaje: {alerta['mensaje']}")
                print(f"   Acción: {alerta['accion']}")
        else:
            print("\n✅ No hay alertas activas")
        
        return todas_alertas

# Crear sistema de alertas
sistema_alertas = SistemaAlertas()

# Evaluar datos con drift
accuracy_con_drift = (df_drift['target'] == df_drift['prediction']).mean()
alertas = sistema_alertas.generar_reporte(resultados_psi, accuracy_con_drift)

## Celda 8: Estrategia de Rollback

Implementaremos un sistema de versionado y rollback para modelos en producción.

In [ ]:
# Celda 8: Rollback strategy

import os
import pickle
import json

class RollbackManager:
    """
    Gestiona versiones de modelos y permite rollback.
    """
    
    def __init__(self, directorio_modelos='modelos/'):
        self.directorio = directorio_modelos
        os.makedirs(directorio_modelos, exist_ok=True)
        self.versiones = self._cargar_versiones()
    
    def _cargar_versiones(self):
        """Carga el registro de versiones."""
        path = os.path.join(self.directorio, 'versiones.json')
        if os.path.exists(path):
            with open(path, 'r') as f:
                return json.load(f)
        return {}
    
    def _guardar_versiones(self):
        """Guarda el registro de versiones."""
        path = os.path.join(self.directorio, 'versiones.json')
        with open(path, 'w') as f:
            json.dump(self.versiones, f, indent=2)
    
    def guardar_version(self, modelo, metricas, version_tag=None):
        """
        Guarda una versión del modelo.
        """
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        version = version_tag or f"v_{timestamp}"
        
        # Guardar modelo
        modelo_path = os.path.join(self.directorio, f'{version}_model.pkl')
        with open(modelo_path, 'wb') as f:
            pickle.dump(modelo, f)
        
        # Registrar versión
        self.versiones[version] = {
            'timestamp': timestamp,
            'metricas': metricas,
            'archivo': modelo_path,
            'estado': 'activo'
        }
        
        self._guardar_versiones()
        print(f"✅ Versión {version} guardada")
        
        return version
    
    def rollback(self, version_objetivo):
        """
        Realiza rollback a una versión específica.
        """
        if version_objetivo not in self.versiones:
            print(f"❌ Versión {version_objetivo} no encontrada")
            return None
        
        # Cargar modelo
        modelo_path = self.versiones[version_objetivo]['archivo']
        with open(modelo_path, 'rb') as f:
            modelo = pickle.load(f)
        
        # Actualizar estado
        for v in self.versiones:
            self.versiones[v]['estado'] = 'inactivo'
        self.versiones[version_objetivo]['estado'] = 'activo'
        
        self._guardar_versiones()
        print(f"✅ Rollback exitoso a versión {version_objetivo}")
        
        return modelo
    
    def listar_versiones(self):
        """
        Lista todas las versiones disponibles.
        """
        print("\n📦 Versiones disponibles:")
        print("-"*50)
        
        for version, info in self.versiones.items():
            estado = '🟢' if info['estado'] == 'activo' else '⚪'
            accuracy = info['metricas'].get('accuracy', 'N/A')
            if isinstance(accuracy, float):
                accuracy = f"{accuracy:.2%}"
            print(f"{estado} {version}: Accuracy = {accuracy}")

# Demostración del sistema de rollback
print("="*60)
print("🔄 SISTEMA DE ROLLBACK")
print("="*60)

# Crear manager
rollback_manager = RollbackManager('modelos_demo/')

# Simular 3 versiones de modelo
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# Versión 1: Modelo bueno
modelo_v1 = LogisticRegression(random_state=42)
modelo_v1.fit(df_original[['feature_1', 'feature_2', 'feature_3']], df_original['target'])
accuracy_v1 = (df_original['target'] == modelo_v1.predict(df_original[['feature_1', 'feature_2', 'feature_3']])).mean()
rollback_manager.guardar_version(modelo_v1, {'accuracy': accuracy_v1}, 'v1_bueno')

# Versión 2: Modelo regular
modelo_v2 = LogisticRegression(C=0.1, random_state=42)
modelo_v2.fit(df_original[['feature_1', 'feature_2', 'feature_3']], df_original['target'])
accuracy_v2 = accuracy_v1 * 0.95  # Simular pequeño decaimiento
rollback_manager.guardar_version(modelo_v2, {'accuracy': accuracy_v2}, 'v2_regular')

# Versión 3: Modelo malo (con concept drift)
modelo_v3 = LogisticRegression(C=0.01, random_state=42)
modelo_v3.fit(df_original[['feature_1', 'feature_2', 'feature_3']], df_original['target'])
accuracy_v3 = (df_drift['target'] == modelo_v3.predict(df_drift[['feature_1', 'feature_2', 'feature_3']])).mean()
rollback_manager.guardar_version(modelo_v3, {'accuracy': accuracy_v3}, 'v3_malo')

# Listar versiones
rollback_manager.listar_versiones()

# Rollback a versión buena
print("\n🔄 Iniciando rollback a v1_bueno...")
modelo_recuperado = rollback_manager.rollback('v1_bueno')

# Verificar recuperación
if modelo_recuperado:
    accuracy_recuperado = (df_original['target'] == modelo_recuperado.predict(df_original[['feature_1', 'feature_2', 'feature_3']])).mean()
    print(f"\n📊 Accuracy después de rollback: {accuracy_recuperado:.2%}")

## Celda 9: Ética - Monitoreo de Sesgos

Implementaremos un sistema para detectar y monitorear sesgos en las predicciones del modelo.

In [ ]:
# Celda 9: Ética - monitoreo de sesgos

class MonitorSesgos:
    """
    Monitorea sesgos en modelos de producción.
    """
    
    def __init__(self, sensitive_features):
        self.sensitive_features = sensitive_features
        self.historial_sesgos = []
    
    def calcular_metricas_equidad(self, X, y_real, y_pred):
        """
        Calcula métricas de equidad por grupo.
        """
        resultados = {}
        
        for feature in self.sensitive_features:
            if feature not in X.columns:
                continue
            
            grupos = X[feature].unique()
            metricas_grupo = {}
            
            for grupo in grupos:
                mascara = X[feature] == grupo
                y_grupo = y_real[mascara]
                pred_grupo = y_pred[mascara]
                
                # Métricas por grupo
                tasa_positivos = np.mean(pred_grupo)
                tasa_verdaderos_positivos = np.mean(pred_grupo[y_grupo == 1]) if np.sum(y_grupo == 1) > 0 else 0
                tasa_falsos_positivos = np.mean(pred_grupo[y_grupo == 0]) if np.sum(y_grupo == 0) > 0 else 0
                
                metricas_grupo[grupo] = {
                    'tasa_positivos': tasa_positivos,
                    'tvp': tasa_verdaderos_positivos,
                    'tfp': tasa_falsos_positivos,
                    'n_samples': int(np.sum(mascara))
                }
            
            # Calcular disparate impact
            tasas = [m['tasa_positivos'] for m in metricas_grupo.values()]
            disparate_impact = min(tasas) / max(tasas) if max(tasas) > 0 else 1
            
            resultados[feature] = {
                'metricas_por_grupo': metricas_grupo,
                'disparate_impact': disparate_impact,
                'alerta_sesgo': disparate_impact < 0.8
            }
        
        return resultados
    
    def generar_reporte_sesgo(self, X, y_real, y_pred, nombre_dataset='Dataset'):
        """
        Genera reporte completo de sesgos.
        """
        metricas = self.calcular_metricas_equidad(X, y_real, y_pred)
        self.historial_sesgos.append({
            'timestamp': datetime.now().isoformat(),
            'dataset': nombre_dataset,
            'metricas': metricas
        })
        
        print("="*60)
        print("⚖️  REPORTE DE EQUIDAD Y SESGOS")
        print("="*60)
        print(f"📊 Dataset: {nombre_dataset}")
        print(f"📅 Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        
        for feature, datos in metricas.items():
            print(f"\n{'='*40}")
            print(f"🔍 Característica sensible: {feature.upper()}")
            print(f"{'='*40}")
            
            # Tabla de métricas por grupo
            print(f"\n{'Grupo':<15} {'Tasa Pos.':<12} {'TVP':<12} {'TFP':<12} {'N':<10}")
            print("-"*60)
            
            for grupo, m in datos['metricas_por_grupo'].items():
                print(f"{grupo:<15} {m['tasa_positivos']:<12.3f} {m['tvp']:<12.3f} {m['tfp']:<12.3f} {m['n_samples']:<10}")
            
            # Disparate Impact
            emoji = '🔴' if datos['alerta_sesgo'] else '🟢'
            print(f"\n{emoji} Disparate Impact Ratio: {datos['disparate_impact']:.3f}")
            print(f"   Umbral aceptable: 0.8 - 1.25")
            print(f"   Estado: {'⚠️  SESGO DETECTADO' if datos['alerta_sesgo'] else '✅ ACEPTABLE'}")
        
        return metricas

# Crear datos con sesgo potencial
np.random.seed(42)
n_samples = 2000

# Datos con características sensibles
X_sesgo = pd.DataFrame({
    'feature_1': np.random.normal(100, 15, n_samples),
    'feature_2': np.random.uniform(10, 100, n_samples),
    'genero': np.random.choice(['M', 'F', 'Otro'], n_samples, p=[0.45, 0.45, 0.1]),
    'grupo_edad': np.random.choice(['18-30', '31-50', '51+'], n_samples, p=[0.3, 0.5, 0.2])
})

# Variable objetivo con sesgo incorporado (inconsciente)
# El modelo aprende que ciertos grupos tienen menor probabilidad
prob_base = 0.5
sesgo_genero = np.where(X_sesgo['genero'] == 'M', 0.1, 
                         np.where(X_sesgo['genero'] == 'F', -0.1, -0.15))
sesgo_edad = np.where(X_sesgo['grupo_edad'] == '31-50', 0.1,
                      np.where(X_sesgo['grupo_edad'] == '18-30', -0.05, -0.15))

probabilidad = prob_base + sesgo_genero + sesgo_edad + 0.1 * (X_sesgo['feature_2'] - 50) / 50
probabilidad = np.clip(probabilidad, 0.1, 0.9)
y_sesgo = np.random.binomial(1, probabilidad)

# Entrenar modelo (que amplificará el sesgo)
from sklearn.ensemble import GradientBoostingClassifier

modelo_sesgo = GradientBoostingClassifier(n_estimators=100, random_state=42)
modelo_sesgo.fit(X_sesgo[['feature_1', 'feature_2']], y_sesgo)
y_pred_sesgo = modelo_sesgo.predict(X_sesgo[['feature_1', 'feature_2']])

# Monitorear sesgos
monitor = MonitorSesgos(['genero', 'grupo_edad'])
reporte = monitor.generar_reporte_sesgo(X_sesgo, y_sesgo, y_pred_sesgo, 'Datos de Producción')

# Resumen ético
print("\n" + "="*60)
print("📋 RESUMEN ÉTICO")
print("="*60)

sesgos_detectados = [f for f, d in reporte.items() if d['alerta_sesgo']]
if sesgos_detectados:
    print(f"\n⚠️  SESGOS DETECTADOS en: {', '.join(sesgos_detectados)}")
    print("\nAcciones recomendadas:")
    print("  1. Revisar datos de entrenamiento")
    print("  2. Considerar técnicas de re-muestreo")
    print("  3. Aplicar regularización de equidad")
    print("  4. Implementar post-procesamiento de equidad")
    print("  5. Documentar hallazgos para auditoría")
else:
    print("\n✅ No se detectaron sesgos significativos")
    print("\nRecomendaciones:")
    print("  1. Continuar monitoreo periódico")
    print("  2. Documentar métricas de equidad")
    print("  3. Revisar con nuevos datos")